Q1. Build Your Personalized Knowledge Base

In [1]:
import pandas as pd

# Fixed FAQ entries given in the assignment
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Last two digits of roll number
roll_number = "1024170226"
last_two_digits = [int(digit) for digit in roll_number[-2:]]

# Category mapping
category_list = ["billing", "account", "general"]

# Personalized entry for digit 2 -> general
digit = last_two_digits[0]
category = category_list[digit % 3]

personalized_entry_1 = {
    "question": "how can i get help with a general query",
    "answer": "You can contact our support team for help with general queries.",
    "keywords": "help support query",
    "category": category
}

# Personalized entry for digit 6 -> billing
digit = last_two_digits[1]
category = category_list[digit % 3]

personalized_entry_2 = {
    "question": "how can i check my fee payment",
    "answer": "You can check your fee payment status through your account.",
    "keywords": "payment fee bill",
    "category": category
}

# Combine all entries
all_entries = fixed_entries + [
    personalized_entry_1,
    personalized_entry_2
]

faq_df = pd.DataFrame(all_entries)

print("Roll Number:", roll_number)
print("Last Two Digits:", last_two_digits)
print("\nFinal FAQ DataFrame:")
faq_df

Roll Number: 1024170226
Last Two Digits: [2, 6]

Final FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i get help with a general query,You can contact our support team for help with...,help support query,general
5,how can i check my fee payment,You can check your fee payment status through ...,payment fee bill,billing


Q2. Generate and Score a Hypothesis

In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words.intersection(keywords))

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    return pd.DataFrame(results)

In [4]:
query = input("Enter your question: ")

results = score_query(query, faq_df)

print("\nMatching FAQ entries:")
results


Matching FAQ entries:


,index,question,answer,category,confidence
0,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,2
1,0,what is the annual fee,The annual fee is Rs 500.,billing,1
2,5,how can i check my fee payment,You can check your fee payment status through ...,billing,1


Q3. Find FAQs from the Same Category

In [5]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

In [6]:
personalized_category = personalized_entry_1["category"]

print("Personalized Entry Category:", personalized_category)
print("\nFAQs in the same category:")

same_category(personalized_category, faq_df)

Personalized Entry Category: general

FAQs in the same category:


,question,answer,keywords,category
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
4,how can i get help with a general query,You can contact our support team for help with...,help support query,general


Q4. Add a New Keyword and Save the FAQ Data

In [7]:
entry_index = 0

print("Selected question:")
print(faq_df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

faq_df.loc[entry_index, "keywords"] = (
    faq_df.loc[entry_index, "keywords"] + " " + new_keyword
)

print("\nUpdated entry:")
print(faq_df.loc[entry_index])

Selected question:
what is the annual fee

Updated entry:
question                what is the annual fee
answer               The annual fee is Rs 500.
keywords    fee cost price charge subscription
category                               billing
Name: 0, dtype: object


In [8]:
output_file = "1024170226_faq_data.csv"

faq_df.to_csv(output_file, index=False)

print(f"\nFAQ data saved successfully as {output_file}")


FAQ data saved successfully as 1024170226_faq_data.csv


Q5. FAQ Count Per Category

In [9]:
category_counts = faq_df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_counts)

Number of FAQ entries per category:
category
account    1
billing    3
general    2
dtype: int64


Q6. Improved Scoring Function with Tie Handling

In [10]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words.intersection(keywords))

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    if not results:
        print("No matching FAQ entries found.")
        return pd.DataFrame()

    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    highest_score = results[0]["confidence"]

    top_matches = [
        result for result in results
        if result["confidence"] == highest_score
    ]

    if len(top_matches) > 1:
        print("Tie detected! Multiple FAQ entries have the highest confidence score:")
    else:
        print("Unique highest-confidence match:")

    return pd.DataFrame(top_matches)

In [11]:
tie_query = "fee"

print("Query:", tie_query)

tie_result = score_query_with_ties(tie_query, faq_df)

tie_result

Query: fee
Tie detected! Multiple FAQ entries have the highest confidence score:


,index,question,answer,category,confidence
0,0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1
2,5,how can i check my fee payment,You can check your fee payment status through ...,billing,1


In [12]:
non_tie_query = "password"

print("Query:", non_tie_query)

non_tie_result = score_query_with_ties(non_tie_query, faq_df)

non_tie_result

Query: password
Unique highest-confidence match:


,index,question,answer,category,confidence
0,1,how to reset password,Go to Settings > Reset Password.,account,1
